### PROP_ADAIR: Force and IMU data alignment and subtitle creation
    

This notebook will allow us to capture all of the video, force, and angle data collected for the PROP_ADAIR field trial field excavation, combine it in a meaningful way, allowing us to analyze the force profile as time series data.

To get started, lets import a few modules

In [1]:
import process
import numpy as np
import pandas as pd
import datetime
import os
import sys

In [2]:
sys.version

'3.10.4 (main, Jun 29 2022, 12:14:53) [GCC 11.2.0]'

In [3]:
np.diag((1, 1, 1, 1))

array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]])

In [4]:
time_stamps = pd.read_csv('../video_time_stamps.csv')
time_stamps.head()

FileNotFoundError: [Errno 2] No such file or directory: '../video_time_stamps.csv'

The data frame `time_stamps` includes time signatures from each video that record the start time of the force gauge and the IMU sensor for each tree excavtation that we have video record of. For a few of the data points, we lost some sample data due to either the sensors temporarily disconnecting or because the winch strap malfunctioned, etc. For these few samples we will need to clean the data by splicing the videos together precisely at the point of failure in order to align the IMU and the force gauge data. More on that later...


Each timestamp is recorded in `%h:%m:%s.%f` format, corresponding to hours, minutes, seconds, and microseconds elapsed since the start of the recording. We will use these timestamps to correct the relative time stamps from the force gauge data, and the absolute timestamps from the IMU sensor data and coerce them into a compatible format that corresponds to to the elapsed time in the video. In this way, we can know the exact force being recorded and the angle of the tree for each timepoint in the video.

Lets define a few functions to help out with preprocessing our data.
First, `convert_timestamp` will take a time stame in `%h:%m:%s.%f` format and convert it to an appropriate datetime object

Next, `extract_angle_cleaned` will take a filename of an IMU data file along with the timestamp from the previous function and clean up the time signatures, changing them from absolute signatures to timestamps relative to the video.

Finally, we will resample and interpolate the data based on a specific sampling interval in `resample_time` so that we can merge the IMU sample data with the data generated by the force gauge.

In [4]:
def convert_timestamp(timestamp):
    '''
    Pass a timestamp in %h:%m:%s.%f format and convert it to an appropriate datetime object
    '''
    import re
    import datetime
    
    time_split = [int(i) for i in re.split(r'\W', timestamp)]
    today = datetime.date.today()#get current date, just needed to construct the datettime object
    date_list = [today.year, today.month, today.day]
    
    return datetime.datetime(*(date_list + time_split))

In [5]:
def extract_angle_cleaned(filename, timestamp):
    '''
    Pass a filename and a video timestamp for the relative start time to normalize by
    return a data frame with the adjusted timestamp
    '''
    
    from os.path import exists
    import pandas as pd
    import datetime
    import re
    
    assert type(filename) is str, f"{filename} is not formatted as a string"
    file_ex = exists(filename)
    assert file_ex == True, f"{filename} does not exist in this directory"
    assert type(timestamp) is datetime.datetime, f"{timestamp} is not a datetime.datetime object"
    
    #Read in the the IMU .txt file
    col_names = ['date_time', 'accel', 'a_x', 'a_y', 'a_z', 'velocity', 
                 'v_x', 'v_y', 'v_z', 'angle', 'ang_x', 'ang_y', 'ang_z'] 
    raw_df = pd.read_table(filepath_or_buffer = filename, 
        sep = ",", 
        header = None, 
        names = col_names, 
        usecols = col_names,
        infer_datetime_format = True)
    
    raw_df['date_time'] = pd.to_datetime(raw_df.date_time)
    raw_df['date_time'] = raw_df.date_time - raw_df.date_time.min() + timestamp

    return raw_df

In [6]:
def resample_time(data, interval = '.033333S', on = 'date_time'):
    '''
    Takes a data frame object with a date_time Series and resamples/interpolates according to a specified interval
    '''
    
    import pandas as pd
    
    assert type(data) is pd.core.frame.DataFrame
    assert type(interval) is str
    assert on in data.columns
    
    resample = data.resample(interval, on = 'date_time', origin = 'start').mean()
    intp = resample.interpolate(method = 'linear', limit_direction = 'forward')
    
    intp.reset_index(inplace = True)
    
    return intp

Now we can write some code to help extract the force gauge data. The function `extract_force_profile` takes the filename of the force gauge .csv file, cleans it up, and returns a cleaned data frame object.


In [7]:
def extract_force_profile(filename):
    """
    When passed a filename of a .csv force file
    Returns a data frame of the force in Newtons
    """

    col_names = ['index', 'force_n', 'units', 'seconds', 'time_units']
    force = pd.read_csv(filepath_or_buffer = filename, 
        header = None,
        skiprows = 6,
        index_col = False,
        names = col_names,
        usecols = ['index', 'force_n', 'seconds'])

    return force

The function `convert_secs` takes a float value of time in seconds and converts it to a datetime object representation that is easier for us to work with.

In [8]:
def convert_secs(sec):
    """
    convert a float value of seconds to datetime.datetime object
    Accurate represention of sub 24hr durations in seconds
    """
    import datetime
    
    assert type(sec) is float
    
    total_us = sec*1000000#total microseconds
    us = int(total_us%1000000)#integer microseconds
    s = int(total_us)//1000000#integer seconds
    m = 0#initial value for minutes
    h = 0#initial value for hours

    if s >= 60:
        m += s//60
        s = s%60

    if m >= 60:
        h += m//60
        m = m%60
        
    today = datetime.date.today()#get current date
    date_list = [today.year, today.month, today.day, h, m, s, us]#list containing date elements
    dt = datetime.datetime(*date_list)

    return dt

In [9]:
def normalize_force_time(data, timestamp):
    '''
    When passed an appropriate data frame and timestamp, normalizes the force timestamps to that of the video
    '''
    
    import re
    import pandas as pd
    import numpy as np
    import datetime
    assert type(data) == pd.core.frame.DataFrame
    assert type(timestamp) == datetime.datetime
    
    
    raw = data.copy()
    raw.seconds = raw.seconds.apply(convert_secs)
    min_t = raw.seconds.min()
    
    raw.seconds = raw.seconds + datetime.timedelta(hours = timestamp.hour, 
                                                   minutes = timestamp.minute, 
                                                   seconds = timestamp.second, 
                                                   microseconds = timestamp.microsecond)
    
    raw = raw.rename(columns = {'seconds': 'timestamp'})
    return raw

In [10]:

def get_common_set():
    '''
    gets the intersection of sets of ids for which we have complete force, imu, and video data
    '''

    import os
    vid_list = os.listdir('../blowover_footage/')
    imu_list = os.listdir('../imu_data/')
    force_list = os.listdir('../force_gauge_data')

    for i, v in enumerate(vid_list):
        vid_list[i] = v.replace('.MOV', '')

    for i, v in enumerate(force_list):
        force_list[i] = v.replace('.csv', '')

    for i, v in enumerate(imu_list):
        imu_list[i] = v.replace('.txt', '')

    return {i for i in set(vid_list).intersection(set(imu_list)).intersection(set(force_list))}

Lets create one more function `generate_merges` to bring all of our functionality together to create one merged dataset for each id with complete data

In [11]:
def generate_merges(sample_id, timestamp_df):
    '''
    Takes an id as the argument, returns a 'nearest' merged dataframe containing the force and imu data for each time, t
    '''
    assert sample_id in get_common_set()
    
    
    f_start = convert_timestamp(timestamp_df[timestamp_df.video_id == sample_id].values.item(1))
    imu_start = convert_timestamp(timestamp_df[timestamp_df.video_id == sample_id].values.item(2))
    
    clean_angle = extract_angle_cleaned(filename = ''.join(['../imu_data/', sample_id, '.txt']), timestamp = imu_start)
    imu_resample = resample_time(clean_angle)
    
    clean_force = extract_force_profile(filename=''.join(['../force_gauge_data/', sample_id, '.csv']))
    force_norm = normalize_force_time(clean_force, f_start)
    
    merged = pd.merge_asof(force_norm, imu_resample, left_on = 'timestamp', right_on = 'date_time', direction = 'nearest')
    del merged['date_time']
    
    return merged

Now that we have all the code in place to create merged data files, we can loop through the entire set of ids with complete data files to generate merged dataframes for each id

In [12]:
def generate_srt(df, sample_id = None):
    '''
    Generate an .srt subtitle file for each video
    '''
    import pandas as pd
    
    df1 = df.copy()
    pushed = df1.timestamp#copy timestamp
    pushed.index = pushed.index - 1#roll back the index value
    df2 = pd.merge(pushed, df1, left_index = True, right_index = True, how = 'outer')#merge on the index(lags the timestamp by one t)
    df2 = df2.rename(columns = {'timestamp_x': 't_end', 'timestamp_y': 't_start'})
    df2.drop([-1], inplace = True)#Drop ends
    df2.drop([df2.shape[0]-1], inplace = True)
    
    df2[['NONE', 't_start']] = df2['t_start'].astype('str').str.split(expand = True)
    df2[['NONE', 't_end']] = df2['t_end'].astype('str').str.split(expand = True)
    del df2['NONE']
    df2['t_start'] = df2['t_start'].str[0:12]#clip trailing zeros
    df2['t_end'] = df2['t_end'].str[0:12]
    
    df2['time_slice'] = df2.t_start + " --> " + df2.t_end#create an srt timestamp string
    df2['time_slice'] = df2['time_slice'].str.replace('.', ',', regex = False)
    
    df2 = df2[['time_slice', 'force_n', 'a_x', 'a_y', 'a_z', 'v_x', 'v_y', 'v_z', 'ang_x', 'ang_y', 'ang_z']]
    
    
    with open(''.join(['./subtitles/', sample_id, '.srt']), 'w') as f:
        for ind, row in enumerate(df2.itertuples()):
            tmp_list = [str(ind), '\n', str(row[1]), '\n', '{\\an7}\n', f"ANGLE: {round(row[10], 1)}", '\n', f'FORCE: {int(row[2])}', '\n\n']
            f.writelines(tmp_list)
    f.close()
    
    return ' '.join([sample_id, 'finished'])

Now let's loop through all of the ids, extract the datasets, merge, and then export .srt files for each video

In [13]:
id_list = [i for i in get_common_set()]
time_stamps = pd.read_excel('../video_time_stamps.xlsx')

for sample_id in id_list:
    merged_df = generate_merges(sample_id, time_stamps)
    generate_srt(merged_df, sample_id)

print('Subtitle Creation Complete')
    

Subtitle Creation Complete
